[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)

# LangChain Academy

欢迎来到 LangChain Academy！

## 背景

在 LangChain，我们致力于让构建 LLM 应用变得简单。你可以构建的一种 LLM 应用是 Agent。围绕构建 Agent 有很多令人兴奋的地方，因为它们可以自动化以前无法实现的广泛任务。

然而在实践中，构建能够可靠执行这些任务的系统非常困难。在我们与用户一起将 Agent 投入生产的过程中，我们了解到通常需要更多的控制。你可能需要 Agent 始终首先调用特定工具，或者根据其状态使用不同的 Prompt。

为了解决这个问题，我们构建了 [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) —— 一个用于构建 Agent 和多 Agent 应用的框架。与 LangChain 包分开，LangGraph 的核心设计理念是帮助开发者在 Agent 工作流中添加更好的精确性和控制力，以应对真实世界系统的复杂性。

## 课程结构

本课程以一组模块的形式组织，每个模块专注于与 LangGraph 相关的特定主题。你会看到每个模块都有一个文件夹，其中包含一系列 notebook。每个 notebook 都配有视频来帮助讲解概念，但 notebook 也是独立的，意味着它们包含解释内容，可以脱离视频独立阅读。每个模块文件夹还包含一个 `studio` 文件夹，其中包含一组可以加载到 [LangSmith Studio](https://docs.langchain.com/langsmith/quick-start-studio) 中的图，这是我们用于构建 LangGraph 应用的 IDE。

## 环境搭建

开始之前，请按照 `README` 中的说明创建虚拟环境并安装依赖。

## 聊天模型

在本课程中，我们将使用聊天模型（Chat Models），它们接收一系列消息作为输入，并返回消息作为输出。LangChain 通过[第三方集成](https://docs.langchain.com/oss/python/integrations/chat)支持多种模型。默认情况下，本课程将使用 [ChatDeepSeek](https://python.langchain.com/docs/integrations/chat/deepseek/)，因为它既流行又性能出色。如前所述，请确保你已设置 `DEEPSEEK_API_KEY`。

让我们检查你的 `DEEPSEEK_API_KEY` 是否已设置，如果未设置，系统会要求你输入。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_deepseek langchain_core langchain_community langchain-tavily

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

[这里](https://docs.langchain.com/oss/python/langchain/models)是一个有用的指南，介绍了你可以用聊天模型做的所有事情，但下面我们展示一些重点功能。如果你已按照 README 中的说明运行了 `pip install -r requirements.txt`，那么你已经安装了 `langchain-deepseek` 包。有了这个包，我们就可以实例化 `ChatDeepSeek` 模型对象。你可以在[这里](https://deepseek.com/api/pricing/)查看各种模型的定价。Notebook 默认使用 `deepseek-chat`，因为它在质量、价格和速度之间提供了良好的平衡，但你也可以选择价格更低的 `deepseek-chat` 系列或更新的模型。

聊天模型有几个[标准参数](https://docs.langchain.com/oss/python/langchain/models#parameters)可以设置。最常见的两个是：

* `model`：模型名称
* `temperature`：采样温度

`Temperature` 控制模型输出的随机性或创造性，低温（接近 0）会产生更确定和聚焦的输出。这适用于需要准确性或事实性回答的任务。高温（接近 1）适用于创造性任务或生成多样化的回复。

In [ ]:
from langchain_deepseek import ChatDeepSeek
gpt4o_chat = ChatDeepSeek(model="deepseek-chat", temperature=0)
gpt35_chat = ChatDeepSeek(model="deepseek-chat", temperature=0)

LangChain 中的聊天模型有许多[默认方法](https://reference.langchain.com/python/langchain_core/runnables)。在大多数情况下，我们将使用：

* [stream](https://docs.langchain.com/oss/python/langchain/models#stream)：流式返回响应的数据块
* [invoke](https://docs.langchain.com/oss/python/langchain/models#invoke)：对输入调用链

并且，如前所述，聊天模型接收[消息](https://docs.langchain.com/oss/python/langchain/messages)作为输入。消息有一个角色（描述谁在说这条消息）和一个内容属性。我们稍后会详细讨论这个，但这里先展示基础知识。

In [ ]:
from langchain_core.messages import HumanMessage

# 创建一条消息
msg = HumanMessage(content="Hello world", name="Lance")

# 消息列表
messages = [msg]

# 用消息列表调用模型
gpt4o_chat.invoke(messages)

我们得到一个 `AIMessage` 响应。另外，注意我们也可以直接传入一个字符串来调用聊天模型。当字符串作为输入传入时，它会被转换为 `HumanMessage`，然后传递给底层模型。

In [ ]:
gpt4o_chat.invoke("hello world")

In [ ]:
gpt35_chat.invoke("hello world")

所有聊天模型的接口是一致的，模型通常在每个 notebook 启动时初始化一次。

因此，如果你对另一个提供商有强烈偏好，可以轻松在模型之间切换，而无需更改下游代码。

## 搜索工具

你还会在 README 中看到 [Tavily](https://tavily.com/)，它是一个针对 LLM 和 RAG 优化的搜索引擎，旨在提供高效、快速且持久的搜索结果。如前所述，注册非常简单，并提供慷慨的免费额度。部分课程（模块 4）默认使用 Tavily，但如果你想为自己修改代码，当然也可以使用其他搜索工具。

In [ ]:
_set_env("TAVILY_API_KEY")

In [ ]:
from langchain_tavily import TavilySearch  # 已更新至 1.0

tavily_search = TavilySearch(max_results=3)

data = tavily_search.invoke({"query": "What is LangGraph?"})
search_docs = data.get("results", data)

In [ ]:
search_docs